In [1]:
import os
import pandas as pd
import numpy as np
import rdkit
from rdkit import Chem
from sklearn.model_selection import train_test_split

import torch
from torch_geometric.data import Data, InMemoryDataset
from torch_geometric.loader import DataLoader

In [2]:
def make_smile_canonical(smile):
    """将 SMILES 转换为标准形式，避免重复"""
    try:
        mol = Chem.MolFromSmiles(smile)
        if mol is None:
            return np.nan
        return Chem.MolToSmiles(mol, canonical=True)
    except:
        return np.nan


In [3]:

def inspect_dataset(df: pd.DataFrame, name: str = "Dataset") -> None:
    """
    打印数据集的基本信息，以便检查：
    - 样本总数
    - 每列缺失值数量
    - 数值列摘要统计
    - SMILES 唯一值比例及最常见项
    - 目标列（Tc, Tg, Density）缺失比例
    """
    total = len(df)
    print(f"=== {name} 概览 ===")
    print(f"样本总数: {total}\n")
    
    # 每列缺失值统计
    print("每列缺失值统计:")
    missing = df.isna().sum().sort_values(ascending=False)
    print(missing[missing > 0], "\n")
    
    # 数值列摘要统计
    print("数值列摘要（包括目标属性）:")
    print(df.describe(include=[float]), "\n")
    
    # SMILES 唯一值比例
    if 'SMILES' in df.columns:
        unique_smiles = df['SMILES'].nunique()
        print(f"SMILES 唯一值: {unique_smiles} / {total} ({unique_smiles/total:.2%})\n")
        print("最常见的 10 个 SMILES 及出现次数:")
        print(df['SMILES'].value_counts().head(10), "\n")
    
    # 目标列缺失比例
    for col in ['Tc', 'Tg', 'Density']:
        if col in df.columns:
            miss = df[col].isna().sum()
            print(f"{col} 缺失: {miss} / {total} ({miss/total:.2%})")
    print("========================\n")

In [4]:
def maybe_correct_offset(df_train, df_extra, target, apply_correction=True):
    """
    如果 train 和 extra 中有重复 SMILES，且都有 target 值，
    则计算系统偏移量（extra - train），并在需要时修正 df_extra 中的 target 值。

    Args:
        df_train (pd.DataFrame): 原始训练数据
        df_extra (pd.DataFrame): 外部增强数据
        target (str): 要增强的目标列名（如 'Tg', 'Tc' 等）
        apply_correction (bool): 是否应用偏移校正

    Returns:
        df_extra_new (pd.DataFrame): 修正后的外部数据副本
    """
    df_train = df_train.copy()
    df_extra = df_extra.copy()

    # Canonical SMILES
    df_train['SMILES'] = df_train['SMILES'].apply(make_smile_canonical)
    df_extra['SMILES'] = df_extra['SMILES'].apply(make_smile_canonical)

    # 保留有 target 的样本
    df_train_tc = df_train[df_train[target].notnull()]
    df_extra_tc = df_extra[df_extra[target].notnull()]

    # 找到重复的 SMILES
    common_smiles = set(df_train_tc['SMILES']) & set(df_extra_tc['SMILES'])
    print(f'common_smiles: {len(common_smiles)}')
    # 提取并聚合
    df_train_common = df_train_tc[df_train_tc['SMILES'].isin(common_smiles)].groupby('SMILES')[target].mean()
    df_extra_common = df_extra_tc[df_extra_tc['SMILES'].isin(common_smiles)].groupby('SMILES')[target].mean()

    # 拼接后对齐
    df_common = pd.concat(
        [df_train_common.rename('train_val'), df_extra_common.rename('extra_val')],
        axis=1
    ).dropna()

    print(f"  → 匹配 SMILES 数量: {len(df_common)}")
    if len(df_common) == 0:
        print(f"  ⚠️ 无法计算 {target} 偏移量，跳过修正")
        return df_extra

    offset = (df_common['extra_val'] - df_common['train_val']).mean()
    std_dev = (df_common['extra_val'] - df_common['train_val']).std()
    print(f"  → {target} 偏移量（extra - train）: {offset:.4f} ± {std_dev:.4f}")

    if apply_correction:
        df_extra[target] = df_extra[target] - offset
        print(f"  ✅ 已对 {target} 进行偏移修正")

    return df_extra

In [5]:

def add_extra_data(df_train, df_extra, target):
    """
    将外部数据 df_extra 根据 target 列添加到 df_train
    1. 标准化 SMILES
    2. 根据 SMILES 聚合外部数据
    3. 填充训练集中缺失的样本
    4. 追加外部独有样本
    """
    print(f"  → 正在增强 {target} 数据，共 {len(df_extra)} 条")
    df_train = df_train.copy()
    df_extra = df_extra.copy()
    df_extra['SMILES'] = df_extra['SMILES'].apply(make_smile_canonical)
    df_extra = df_extra.groupby('SMILES', as_index=False)[target].mean()
    cross_smiles = set(df_extra['SMILES']) & set(df_train['SMILES'])
    print(f'cross_smiles: {len(cross_smiles)}')
    existing = set(df_train[df_train[target].notnull()]['SMILES'])
    cross_smiles -= existing

    for smi in cross_smiles:
        val = df_extra.loc[df_extra['SMILES'] == smi, target].values[0]
        df_train.loc[df_train['SMILES'] == smi, target] = val

    unique_extra = df_extra[~df_extra['SMILES'].isin(df_train['SMILES'])]
    print(f"    填充已有样本 {len(cross_smiles)} 条，新增样本 {len(unique_extra)} 条")
    df_train = pd.concat([df_train, unique_extra], ignore_index=True)
    return df_train

In [6]:
train = pd.read_csv("neurips-open-polymer-prediction-2025/train.csv")

In [7]:
inspect_dataset(train)

=== Dataset 概览 ===
样本总数: 7973

每列缺失值统计:
Tg         7462
Density    7360
Rg         7359
Tc         7236
FFV         943
dtype: int64 

数值列摘要（包括目标属性）:
               Tg          FFV          Tc     Density          Rg
count  511.000000  7030.000000  737.000000  613.000000  614.000000
mean    96.452314     0.367212    0.256334    0.985484   16.419787
std    111.228279     0.029609    0.089538    0.146189    4.608640
min   -148.029738     0.226992    0.046500    0.748691    9.728355
25%     13.674509     0.349549    0.186000    0.890243   12.540328
50%     74.040183     0.364264    0.236000    0.948193   15.052194
75%    161.147595     0.380790    0.330500    1.062096   20.411067
max    472.250000     0.777097    0.524000    1.840999   34.672906 

SMILES 唯一值: 7973 / 7973 (100.00%)

最常见的 10 个 SMILES 及出现次数:
SMILES
*c1ccc(OCCCCCCCCCCCOC(=O)CCCCC(=O)OCCCCCCCCCCCOc2ccc(-c3nnc(*)s3)cc2)cc1                                                                  1
*CC(*)c1ccccc1C(=O)OCCCCCC             

In [8]:
train['SMILES'] = train['SMILES'].apply(make_smile_canonical)

In [9]:
inspect_dataset(train)

=== Dataset 概览 ===
样本总数: 7973

每列缺失值统计:
Tg         7462
Density    7360
Rg         7359
Tc         7236
FFV         943
dtype: int64 

数值列摘要（包括目标属性）:
               Tg          FFV          Tc     Density          Rg
count  511.000000  7030.000000  737.000000  613.000000  614.000000
mean    96.452314     0.367212    0.256334    0.985484   16.419787
std    111.228279     0.029609    0.089538    0.146189    4.608640
min   -148.029738     0.226992    0.046500    0.748691    9.728355
25%     13.674509     0.349549    0.186000    0.890243   12.540328
50%     74.040183     0.364264    0.236000    0.948193   15.052194
75%    161.147595     0.380790    0.330500    1.062096   20.411067
max    472.250000     0.777097    0.524000    1.840999   34.672906 

SMILES 唯一值: 7973 / 7973 (100.00%)

最常见的 10 个 SMILES 及出现次数:
SMILES
*c1ccc(OCCCCCCCCCCCOC(=O)CCCCC(=O)OCCCCCCCCCCCOc2ccc(-c3nnc(*)s3)cc2)cc1                                                                  1
*CC(*)c1ccccc1C(=O)OCCCCCC             

In [10]:
base_path = "neurips-open-polymer-prediction-2025"
extra_dir = os.path.join(base_path, 'smiles-extra-data')

In [11]:
# 2. 增强 Tc
tc_path = os.path.join(base_path, 'Tc_SMILES.csv')
if os.path.exists(tc_path):
    df_tc = pd.read_csv(tc_path).rename(columns={'TC_mean': 'Tc'})
    df_tc = maybe_correct_offset(train, df_tc, 'Tc', apply_correction=False)
    train = add_extra_data(train, df_tc, 'Tc')
else:
    print("  ⚠️ 未找到 Tc 外部数据")

common_smiles: 737
  → 匹配 SMILES 数量: 737
  → Tc 偏移量（extra - train）: -0.0000 ± 0.0003
  → 正在增强 Tc 数据，共 874 条
cross_smiles: 737
    填充已有样本 0 条，新增样本 129 条


In [12]:
tg1_path = os.path.join(extra_dir, 'JCIM_sup_bigsmiles.csv')
if os.path.exists(tg1_path):
    df_tg1 = pd.read_csv(tg1_path, usecols=['SMILES', 'Tg (C)']).rename(columns={'Tg (C)': 'Tg'})
    df_tg1 = maybe_correct_offset(train, df_tg1, 'Tg', apply_correction=False)
    train = add_extra_data(train, df_tg1, 'Tg')
else:
    print("  ⚠️ 未找到 Tg 来源1 数据")

common_smiles: 511
  → 匹配 SMILES 数量: 511
  → Tg 偏移量（extra - train）: 0.0000 ± 0.0000
  → 正在增强 Tg 数据，共 662 条
cross_smiles: 526
    填充已有样本 15 条，新增样本 136 条


In [13]:
tg2_path = os.path.join(extra_dir, 'data_tg3.xlsx')
if os.path.exists(tg2_path):
    df_tg2 = pd.read_excel(tg2_path).rename(columns={'Tg [K]': 'Tg'})
    df_tg2['Tg'] = df_tg2['Tg'] - 273.15
    train = add_extra_data(train, df_tg2, 'Tg')
else:
    print("  ⚠️ 未找到 Tg 来源2 数据")

  → 正在增强 Tg 数据，共 501 条
cross_smiles: 0
    填充已有样本 0 条，新增样本 499 条


In [14]:
d_path = os.path.join(extra_dir, 'data_dnst1.xlsx')
if os.path.exists(d_path):
    df_den = pd.read_excel(d_path)
    df_den = df_den.rename(columns={'density(g/cm3)': 'Density'})[['SMILES', 'Density']]
    df_den['Density'] = pd.to_numeric(df_den['Density'], errors='coerce') - 0.118
    train = add_extra_data(train, df_den, 'Density')
else:
    print("  ⚠️ 未找到 Density 外部数据")

  → 正在增强 Density 数据，共 787 条
cross_smiles: 254
    填充已有样本 110 条，新增样本 525 条


[08:36:11] SMILES Parse Error: syntax error while parsing: *O[Si](*)([R])[R]
[08:36:11] SMILES Parse Error: Failed parsing SMILES '*O[Si](*)([R])[R]' for input: '*O[Si](*)([R])[R]'
[08:36:11] SMILES Parse Error: syntax error while parsing: *NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4
[08:36:11] SMILES Parse Error: Failed parsing SMILES '*NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4' for input: '*NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4'
[08:36:11] SMILES Parse Error: syntax error while parsing: O=C=N[R1]N=C=O.O[R2]O.O[R3]O
[08:36:11] SMILES Parse Error: Failed parsing SMILES 'O=C=N[R1]N=C=O.O[R2]O.O[R3]O' for input: 'O=C=N[R1]N=C=O.O[R2]O.O[R3]O'
[08:36:11] SMILES Parse Error: syntax error while parsing: *CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O
[08:36:11] SMILES Parse Error: Failed parsing SMILES '*CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O' for input: '*CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O'
[08:36:11] SMILES Parse 

In [16]:
# -----------------------------------------------
#  Zero-dependency featurize(): RDKit + Mordred
#  - 取 0/1/2 D 描述符（ignore_3D=True）
#  - 解析失败行 → 全 0     float32 矩阵
# -----------------------------------------------
import time
from rdkit import Chem
from mordred import Calculator, descriptors
import pandas as pd, numpy as np
from tqdm import tqdm
from multiprocessing import cpu_count
from datetime import timedelta

# ① 只这一行即可：忽略 3 D 描述符
_calc = Calculator(descriptors, ignore_3D=True)

def _safe_mol(s):
    try:
        return Chem.MolFromSmiles(s)
    except Exception:
        return None

def featurize(smiles, n_jobs: int = 4, verbose: bool = True) -> np.ndarray:
    """
    smiles : pd.Series 或 list[str]
    returns: ndarray (n_samples, n_features)  float32
    """
    smiles = pd.Series(smiles).fillna('').astype(str).str.strip()
    
    # 让 Mordred 自己开多进程：设置环境变量
    import os
    if n_jobs == -1:
        n_jobs = max(1, cpu_count() - 1)
    os.environ["OMP_NUM_THREADS"] = str(n_jobs)
    
    # 1) RDKit Mol
    mols = [_safe_mol(s) for s in tqdm(smiles,
                                       desc="RDKit Mol",
                                       mininterval=1)]
    
    # 2) Mordred → DataFrame
    df = (_calc.pandas(mols)
                .replace([np.inf, -np.inf], np.nan)
                .astype(np.float32)
                .fillna(0.0))
    
    if verbose:
        print(f"✅ featurize: {df.shape[0]} samples × {df.shape[1]} features")
    return df.values

In [3]:
from xgboost import XGBRegressor
from sklearn.model_selection import KFold
import optuna, numpy as np, pandas as pd
np.seterr(over='ignore') 


X_train = featurize(train['SMILES'])
y_train = train['FFV'].values


print("🔎  X_train 形状:", X_train.shape)
print("🔎  y_train  形状:", y_train.shape)
print("dtype:", X_train.dtype)

print("NaN 计数 ⟶", np.isnan(X_train).sum())
print("Inf 计数 ⟶", np.isinf(X_train).sum())
print("y NaN   ⟶", np.isnan(y_train).sum())

NameError: name 'featurize' is not defined

In [18]:
# -------- 保存 --------
np.savez_compressed(
    "ffv_features.npz",
    X=X_train.astype(np.float32),   # 建议 float32 即可
    y=y_train.astype(np.float32)
)

In [13]:
# -------- 读取 --------
data   = np.load("ffv_features.npz")
X_load = data["X"]
y_load = data["y"]
print(X_load.shape, y_load.shape)

(9262, 1613) (9262,)


In [14]:
mask   = ~np.isnan(y_load)
X_used = X_load[mask]
y_used = y_load[mask]

print("训练集维度:", X_used.shape, y_used.shape)  # (7030, 1613) (7030,)

训练集维度: (7030, 1613) (7030,)


In [ ]:
storage_uri = "sqlite:///ffv_optuna.db"
# ① 建/载 study （文件自动生成）
study = optuna.create_study(
    study_name="ffv_xgb_search",
    direction="minimize",
    storage=storage_uri,
    load_if_exists=True,          # 已存在就接着跑
)
def objective(trial: optuna.trial.Trial) -> float:
    params = {
        "n_estimators":     trial.suggest_int("n_estimators",      300, 1500),
        "max_depth":        trial.suggest_int("max_depth",         4,   10),
        "learning_rate":    trial.suggest_float("eta",             1e-3, 0.1, log=True),
        "subsample":        trial.suggest_float("subsample",       0.5,  1.0),
        "colsample_bytree": trial.suggest_float("colsample",       0.5,  1.0),
        "reg_lambda":       trial.suggest_float("reg_lambda",      1e-3, 10, log=True),
        "tree_method":      "hist",
        "n_jobs":           -1,
        "verbosity":        0,
    }

    cv   = KFold(n_splits=3, shuffle=True, random_state=0)
    maes = []

    for fold, (tr_idx, val_idx) in enumerate(
        tqdm(cv.split(X_used),
             total=cv.get_n_splits(),
             desc=f"Trial {trial.number}",
             leave=False)
    ):
        model = XGBRegressor(**params)
        model.fit(
            X_used[tr_idx], y_used[tr_idx],
            verbose=False,
        )
        pred   = model.predict(X_used[val_idx])
        mae    = np.abs(pred - y_used[val_idx]).mean()
        maes.append(mae)

    return float(np.mean(maes))

t0 = time.time()   
def progress_cb(study, trial):
    elapsed = time.time() - t0
    h, m = divmod(elapsed // 60, 60)
    s     = elapsed % 60
    print(f"✅ Trial {trial.number:03d}  MAE={trial.value:.4f}  "
          f"Best={study.best_value:.4f}  "
          f"Elapsed={int(h):02d}:{int(m):02d}:{int(s):02d}")

study.optimize(
    objective,
    n_trials = 50,          # 或 timeout=3600
    callbacks=[progress_cb],
    show_progress_bar=True
)

print("\n🎯 Best MAE :", study.best_value)
print("🏆 Best Params:", study.best_params)

# ───────────────────────────────
# 5. 用最佳参数拟合全部有效样本
# ───────────────────────────────
best_params = study.best_params | {"n_jobs": -1, "tree_method": "hist"}
final_model = XGBRegressor(**best_params).fit(X_used, y_used)

[I 2025-07-30 09:33:27,563] Using an existing study with name 'ffv_xgb_search' instead of creating a new one.


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-07-30 09:37:24,641] Trial 8 finished with value: 0.011460629291832447 and parameters: {'n_estimators': 410, 'max_depth': 9, 'eta': 0.0024283146294632224, 'subsample': 0.8426007857841925, 'colsample': 0.5855486441274111, 'reg_lambda': 0.0044508567270722405}. Best is trial 1 with value: 0.005913501605391502.
✅ Trial 008  MAE=0.0115  Best=0.0059  Elapsed=00:03:57


[I 2025-07-30 09:38:51,237] Trial 9 finished with value: 0.006488300859928131 and parameters: {'n_estimators': 467, 'max_depth': 5, 'eta': 0.03385616075215974, 'subsample': 0.6423666628501425, 'colsample': 0.7083138455825873, 'reg_lambda': 1.0204676488438384}. Best is trial 1 with value: 0.005913501605391502.
✅ Trial 009  MAE=0.0065  Best=0.0059  Elapsed=00:05:23


[I 2025-07-30 09:48:45,443] Trial 10 finished with value: 0.0063659618608653545 and parameters: {'n_estimators': 1176, 'max_depth': 10, 'eta': 0.012010809310489017, 'subsample': 0.8843062635613815, 'colsample': 0.9768927191635459, 'reg_lambda': 0.048387733623310225}. Best is trial 1 with value: 0.005913501605391502.
✅ Trial 010  MAE=0.0064  Best=0.0059  Elapsed=00:15:17


...

[I 2025-07-30 10:17:15,553] Trial 18 finished with value: 0.006020639091730118 and parameters: {'n_estimators': 642, 'max_depth': 6, 'eta': 0.054069888186606865, 'subsample': 0.9936167195589687, 'colsample': 0.6535819984022538, 'reg_lambda': 1.7646034969566844}. Best is trial 1 with value: 0.005913501605391502.
✅ Trial 018  MAE=0.0060  Best=0.0059  Elapsed=00:43:47


[I 2025-07-30 10:23:49,684] Trial 19 finished with value: 0.006055276840925217 and parameters: {'n_estimators': 1023, 'max_depth': 10, 'eta': 0.019402899120021664, 'subsample': 0.5079235625362153, 'colsample': 0.7860858489608931, 'reg_lambda': 2.869526255849418}. Best is trial 1 with value: 0.005913501605391502.
✅ Trial 019  MAE=0.0061  Best=0.0059  Elapsed=00:50:22


[I 2025-07-30 10:29:15,954] Trial 20 finished with value: 0.0070077502168715 and parameters: {'n_estimators': 1239, 'max_depth': 6, 'eta': 0.004568401394679986, 'subsample': 0.768553539759138, 'colsample': 0.6325867059756315, 'reg_lambda': 0.4528760517170669}. Best is trial 1 with value: 0.005913501605391502.
✅ Trial 020  MAE=0.0070  Best=0.0059  Elapsed=00:55:48


[I 2025-07-30 10:31:03,646] Trial 21 finished with value: 0.006755091715604067 and parameters: {'n_estimators': 618, 'max_depth': 5, 'eta': 0.023797340058683544, 'subsample': 0.9900192636791361, 'colsample': 0.5031201650413191, 'reg_lambda': 3.432877550593345}. Best is trial 1 with value: 0.005913501605391502.
✅ Trial 021  MAE=0.0068  Best=0.0059  Elapsed=00:57:36


[I 2025-07-30 10:34:21,236] Trial 22 finished with value: 0.0063003352843225 and parameters: {'n_estimators': 1313, 'max_depth': 7, 'eta': 0.08885996056801113, 'subsample': 0.7053480137168519, 'colsample': 0.8355682273231535, 'reg_lambda': 0.057484024740822634}. Best is trial 1 with value: 0.005913501605391502.
✅ Trial 022  MAE=0.0063  Best=0.0059  Elapsed=01:00:53


In [ ]:
new_path = os.path.join(base_path, 'train_supplement')
ffv_path = os.path.join(new_path, 'dataset4.csv')
if os.path.exists(ffv_path):
    df_ffv = pd.read_csv(ffv_path)
    df_ffv = df_ffv.rename(columns={'FFV': 'FFV'})[['SMILES', 'FFV']]
    df_ffv = maybe_correct_offset(train, df_ffv, 'FFV', apply_correction=False)
    train = add_extra_data(train, df_ffv, 'FFV')
    print(f'add dataset4: {len(train)}')    
else:
    print("  ⚠️ 未找到 Density 外部数据")

common_smiles: 0
  → 匹配 SMILES 数量: 0
  ⚠️ 无法计算 FFV 偏移量，跳过修正
  → 正在增强 FFV 数据，共 862 条
cross_smiles: 43
    填充已有样本 43 条，新增样本 819 条
add dataset4: 10081


In [5]:
storage_uri = "sqlite:///ffv_optuna.db"
# ① 建/载 study （文件自动生成）
study = optuna.create_study(
    study_name="ffv_xgb_search",
    direction="minimize",
    storage=storage_uri,
    load_if_exists=True,          # 已存在就接着跑
)
best_params = study.best_trial.params

[I 2025-07-31 05:14:45,622] Using an existing study with name 'ffv_xgb_search' instead of creating a new one.


In [7]:
#print("Best trial score:", best_trial.value)
print("Best trial params:", best_params)

Best trial params: {'n_estimators': 1144, 'max_depth': 5, 'eta': 0.03619636054577697, 'subsample': 0.651721377419442, 'colsample': 0.5556445678221342, 'reg_lambda': 0.4216666672011093}


In [9]:
model = XGBRegressor(**best_params)
print(model)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample=0.5556445678221342, colsample_bylevel=None,
             colsample_bynode=None, colsample_bytree=None, device=None,
             early_stopping_rounds=None, enable_categorical=False,
             eta=0.03619636054577697, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1144, ...)


In [15]:
model.fit(X_used, y_used, verbose=True)

/root/miniconda3/envs/test1/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [05:17:53] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "colsample" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [17]:
import joblib
joblib.dump(model, "best_xgb_model.pkl")

['best_xgb_model.pkl']

In [18]:
model = joblib.load("best_xgb_model.pkl")

In [20]:
df_test = pd.read_csv("neurips-open-polymer-prediction-2025/train_supplement/dataset4.csv")
df_test = df_test.rename(columns={'FFV': 'FFV'})[['SMILES', 'FFV']]

X_test = df_test['SMILES'].apply(featurize).values
y_test = df_test['FFV'].values

100%|██████████| 1/1 [00:00<00:00,  2.83it/s]


✅ featurize: 1 samples × 1613 features


100%|██████████| 1/1 [00:00<00:00,  1.73it/s]


✅ featurize: 1 samples × 1613 features


100%|██████████| 1/1 [00:00<00:00,  1.06it/s]


...

✅ featurize: 1 samples × 1613 features


100%|██████████| 1/1 [00:00<00:00,  1.72it/s]


✅ featurize: 1 samples × 1613 features


100%|██████████| 1/1 [00:00<00:00,  1.79it/s]

✅ featurize: 1 samples × 1613 features


In [25]:
print(type(X_test))
print(f'shape:{X_test.shape}')
X_test = np.vstack(X_test)

<class 'numpy.ndarray'>
shape:(862,)


In [27]:
X_test.iloc[0] if isinstance(X_test, pd.DataFrame) else X_test[0]

array([  0.      ,   0.      ,   0.      , ..., 277.      ,  13.895833,
        10.      ], dtype=float32)

In [28]:
y_pred = model.predict(X_test)

mae = np.mean(np.abs(y_pred - y_test))
bias = np.mean(y_pred - y_test)

print(f"MAE: {mae:.4f}")
print(f"Bias: {bias:.4f}")

MAE: 0.0048
Bias: 0.0008


In [24]:
df_t = pd.read_csv('neurips-open-polymer-prediction-2025/train.csv')
df_t = df_t[['SMILES', 'FFV']]
df_t.describe()

,FFV
count,7030.000000
mean,0.367212
std,0.029609
min,0.226992
25%,0.349549
50%,0.364264
75%,0.380790
max,0.777097
